# Page 5: few-shot chain of thought

This notebook runs the Qwen2.5 base models (1.5B, 3B, 7B) on 495 MATH-500 problems in two ways:

- **zero-shot**: the problem alone.
- **few-shot**: five worked examples, then the problem. The worked solutions were written for this experiment, expanding MATH's own solutions.

Every reply is appended to `traces.jsonl` on Google Drive as soon as its batch finishes. If the runtime stops, run the notebook again from the top and it continues where it left off. The results are saved beside it as `summary.json` and two charts.

In [ ]:
!pip install -q transformers accelerate datasets "math-verify[antlr4_13_2]"

import gc, hashlib, json, os, random, statistics, time

import matplotlib.pyplot as plt
import torch
from datasets import load_dataset
from math_verify import parse, verify
from transformers import AutoModelForCausalLM, AutoTokenizer

assert torch.cuda.is_available(), "This notebook needs a GPU runtime."
print("GPU:", torch.cuda.get_device_name(0))

## Settings and Google Drive

Outputs go to `MyDrive/page_5_few_shot/`. `LIMIT` runs only the first few test problems, for a quick check; a limited run saves to its own file.

In [ ]:
MODELS = ["Qwen/Qwen2.5-1.5B", "Qwen/Qwen2.5-3B", "Qwen/Qwen2.5-7B"]
ARMS = ["zero_shot", "few_shot"]
MAX_NEW_TOKENS = 1024
BATCH_SIZE = 64          # halved for any batch that runs out of memory
LIMIT = None             # e.g. 20 for a quick check

try:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT_DIR = "/content/drive/MyDrive/page_5_few_shot"
except ImportError:
    OUT_DIR = "out/page_5_few_shot"
if LIMIT:
    OUT_DIR = os.path.join(OUT_DIR, f"limit{LIMIT}")
os.makedirs(OUT_DIR, exist_ok=True)
TRACES = os.path.join(OUT_DIR, "traces.jsonl")
print("Saving to", OUT_DIR)

## Problems and examples

The five example problems are chosen by shuffling MATH-500 with seed 42 and taking the first problem of each level. The other 495 problems are the test set.

In [ ]:
rows = list(load_dataset("HuggingFaceH4/MATH-500", split="test"))

order = list(range(len(rows)))
random.seed(42)
random.shuffle(order)
examples, levels_seen = [], set()
for i in order:
    if rows[i]["level"] not in levels_seen:
        examples.append(rows[i])
        levels_seen.add(rows[i]["level"])
    if len(examples) == 5:
        break

assert [r["unique_id"] for r in examples] == [
    "test/precalculus/697.json", "test/algebra/101.json", "test/geometry/178.json",
    "test/number_theory/769.json", "test/number_theory/612.json",
]

example_ids = {r["unique_id"] for r in examples}
test_rows = [r for r in rows if r["unique_id"] not in example_ids]
if LIMIT:
    test_rows = test_rows[:LIMIT]
print(len(test_rows), "test problems")

## The worked solutions

One worked solution per example problem, each expanding MATH's own solution: the same method, with the steps MATH skips written out. The hash check confirms they are the approved text in `page_5_few_shot_solutions.json`.

In [ ]:
SOLUTIONS = {
    "test/precalculus/697.json": r"""We have that
\begin{align*}
\tan 100^\circ + 4 \sin 100^\circ &= \frac{\sin 100^\circ}{\cos 100^\circ} + 4 \sin 100^\circ \\
&= \frac{\sin 80^\circ}{-\cos 80^\circ} + 4 \sin 80^\circ \\
&= -\frac{\cos 10^\circ}{\sin 10^\circ} + 4 \cos 10^\circ \\
&= \frac{4 \cos 10^\circ \sin 10^\circ - \cos 10^\circ}{\sin 10^\circ}.
\end{align*}The second line uses $\sin(180^\circ - x) = \sin x$ and $\cos(180^\circ - x) = -\cos x$ with $x = 80^\circ$. The third line uses $\sin(90^\circ - x) = \cos x$ and $\cos(90^\circ - x) = \sin x$ with $x = 10^\circ$, so $\sin 80^\circ = \cos 10^\circ$ and $\cos 80^\circ = \sin 10^\circ$. The fourth line writes both terms over the common denominator $\sin 10^\circ$.

By the double angle formula $\sin 2x = 2 \sin x \cos x$, we have $2 \sin 10^\circ \cos 10^\circ = \sin 20^\circ$, so $4 \cos 10^\circ \sin 10^\circ = 2 \sin 20^\circ$. Also, $\cos 10^\circ = \sin 80^\circ$. So
\begin{align*}
\frac{4 \cos 10^\circ \sin 10^\circ - \cos 10^\circ}{\sin 10^\circ} &= \frac{2 \sin 20^\circ - \cos 10^\circ}{\sin 10^\circ} \\
&= \frac{\sin 20^\circ + \sin 20^\circ - \sin 80^\circ}{\sin 10^\circ}.
\end{align*}By sum-to-product, $\sin a - \sin b = 2 \cos \frac{a + b}{2} \sin \frac{a - b}{2}$. With $a = 20^\circ$ and $b = 80^\circ$,
\[\sin 20^\circ - \sin 80^\circ = 2 \cos 50^\circ \sin (-30^\circ) = -\cos 50^\circ,\]since $\sin(-30^\circ) = -\frac{1}{2}$. So
\begin{align*}
\frac{\sin 20^\circ + \sin 20^\circ - \sin 80^\circ}{\sin 10^\circ} &= \frac{\sin 20^\circ - \cos 50^\circ}{\sin 10^\circ} \\
&= \frac{\sin 20^\circ - \sin 40^\circ}{\sin 10^\circ},
\end{align*}using $\cos 50^\circ = \sin (90^\circ - 50^\circ) = \sin 40^\circ$. By sum-to-product again, with $a = 20^\circ$ and $b = 40^\circ$,
\[\sin 20^\circ - \sin 40^\circ = 2 \cos 30^\circ \sin (-10^\circ) = -\sqrt{3} \sin 10^\circ,\]since $\cos 30^\circ = \frac{\sqrt{3}}{2}$ and $\sin(-10^\circ) = -\sin 10^\circ$. Since $\sin 10^\circ \neq 0$, we can cancel it:
\[\frac{\sin 20^\circ - \sin 40^\circ}{\sin 10^\circ} = \frac{-\sqrt{3} \sin 10^\circ}{\sin 10^\circ} = -\sqrt{3}.\]As a check, $\tan 100^\circ \approx -5.6713$ and $4 \sin 100^\circ \approx 3.9392$, which add to about $-1.7321$, and $-\sqrt{3} \approx -1.7321$.

So $\tan 100^\circ + 4 \sin 100^\circ = \boxed{-\sqrt{3}}.$""",
    "test/algebra/101.json": r"""We want all $x$ with $x^2 - 5x - 4 \le 10$. Subtracting $10$ from both sides gives
\[x^2 - 5x - 4 - 10 \le 0,\]which simplifies to
\[x^2 - 5x - 14 \le 0.\]To factor the left-hand side, we look for two numbers whose product is $-14$ and whose sum is $-5$. The pairs of integers with product $-14$ are $1$ and $-14$, $-1$ and $14$, $2$ and $-7$, and $-2$ and $7$, and their sums are $-13$, $13$, $-5$, and $5$. The pair $2$ and $-7$ has sum $-5$, so the left-hand quadratic factors as
\[x^2 - 5x - 14 = (x - 7)(x + 2).\]As a check, $(x - 7)(x + 2) = x^2 + 2x - 7x - 14 = x^2 - 5x - 14$. So the inequality is
\[(x - 7)(x + 2) \le 0.\]A product of two numbers is less than or equal to zero exactly when one of the factors is zero, or when the two factors have opposite signs.

The product is zero when $x - 7 = 0$ or $x + 2 = 0$, that is, when $x = 7$ or $x = -2$. Both of these values satisfy the inequality.

For the factors to have opposite signs, one must be positive and the other negative. Since $(x + 2) - (x - 7) = 9$, the factor $x + 2$ is always the larger of the two, so we need $x + 2 > 0$ and $x - 7 < 0$. The first gives $x > -2$ and the second gives $x < 7$, so $-2 < x < 7$. The other arrangement, $x + 2 < 0$ and $x - 7 > 0$, would need $x < -2$ and $x > 7$ at the same time, which is impossible.

Combining the two cases, the inequality holds exactly when $-2 \le x \le 7$. As a check, $x = 0$ gives $0^2 - 5 \cdot 0 - 4 = -4$, which is at most $10$, while $x = 8$ gives $64 - 40 - 4 = 20$ and $x = -3$ gives $9 + 15 - 4 = 20$, which are both greater than $10$.

In interval notation, $\boxed{x \in [-2,7]}$.""",
    "test/geometry/178.json": r"""The triangle is shown below:

[asy]
pair A,B,C;
A = (0,0);
B = (5,0);
C = (5,10);
draw(A--B--C--A);
draw(rightanglemark(C,B,A,16));
label("$A$",A,SW);
label("$B$",B,SE);
label("$C$",C,N);
[/asy]

Since $\angle B = 90^\circ$, the side $AC$ is the hypotenuse. For the angle $A$, the side opposite $A$ is $BC$ and the side adjacent to $A$ is $AB$. So
\[\sin A = \frac{\text{opposite}}{\text{hypotenuse}} = \frac{BC}{AC} \quad \text{and} \quad \cos A = \frac{\text{adjacent}}{\text{hypotenuse}} = \frac{AB}{AC}.\]Substituting these into $\sin A = 2\cos A$ gives us
\[\frac{BC}{AC} = 2\cdot\frac{AB}{AC}.\]Both fractions have the same denominator $AC$, and $AC$ is a length, so it is not zero. Multiplying both sides by $AC$ gives
\[BC = 2AB.\]Since $AB$ is also a length and not zero, we can divide both sides by $AB$ to get
\[\frac{BC}{AB} = 2.\]The tangent of $A$ is the opposite side divided by the adjacent side:
\[\tan A = \frac{\text{opposite}}{\text{adjacent}} = \frac{BC}{AB},\]and we just found that this ratio is $2$.

We can check this in two ways. First, $\tan A = \frac{\sin A}{\cos A}$, and substituting $\sin A = 2\cos A$ gives $\tan A = \frac{2\cos A}{\cos A}$. Since $A$ is an acute angle of a right triangle, $\cos A > 0$, so we can cancel $\cos A$ and get $\tan A = 2$, which agrees. Second, in the triangle shown, $AB = 5$ and $BC = 10$, so $BC = 2AB$. Then $AC = \sqrt{5^2 + 10^2} = \sqrt{125} = 5\sqrt{5}$, so $\sin A = \frac{10}{5\sqrt{5}} = \frac{2}{\sqrt{5}}$ and $\cos A = \frac{5}{5\sqrt{5}} = \frac{1}{\sqrt{5}}$, and indeed $\sin A = 2\cos A$.

Therefore, $\tan A = \boxed{2}$.""",
    "test/number_theory/769.json": r"""We use the property that $a \equiv b \pmod{m}$ implies $a^c \equiv b^c \pmod{m}$, so we can replace each base by a smaller number with the same remainder.

First we reduce the bases modulo $11$. Since $11 \cdot 12 = 132$, we have $129 = 132 - 3$, so $129 \equiv -3 \pmod{11}$. Since $11 \cdot 9 = 99$, we have $96 = 99 - 3$, so $96 \equiv -3 \pmod{11}$. Therefore
\[129^{34}+96^{38} \equiv (-3)^{34}+(-3)^{38} \pmod{11}.\]Both exponents are even, so the minus signs disappear: $(-3)^{34} = 3^{34}$ and $(-3)^{38} = 3^{38}$. So
\[129^{34}+96^{38} \equiv 3^{34}+3^{38} \pmod{11}.\]Next we look for a small power of $3$ that is $1$ modulo $11$. Computing powers of $3$ one at a time and reducing modulo $11$ at each step,
\begin{align*}
3^1 &= 3, \\
3^2 &= 9, \\
3^3 &= 27 = 2 \cdot 11 + 5 \equiv 5 \pmod{11}, \\
3^4 &\equiv 3 \cdot 5 = 15 = 11 + 4 \equiv 4 \pmod{11}, \\
3^5 &\equiv 3 \cdot 4 = 12 = 11 + 1 \equiv 1 \pmod{11}.
\end{align*}So $3^5 \equiv 1 \pmod{11}$. This means every power $(3^5)^k$ is also $1$ modulo $11$, so we split each exponent into a multiple of $5$ plus a remainder. Since $34 = 5 \cdot 6 + 4$, we can see that $3^{34} = (3^5)^{6} \cdot 3^4$. Since $38 = 5 \cdot 7 + 3$, we have $3^{38} = (3^5)^{7} \cdot 3^3$.

Then,
\begin{align*}
129^{34}+96^{38}&\equiv (3^5)^{6} \cdot 3^4 + (3^5)^{7} \cdot 3^3\\
& \equiv 1^{6} \cdot 3^4 + 1^{7} \cdot 3^3\\
& \equiv 3^4 + 3^3\\
& \equiv 81 + 27\\
& \equiv 108 \pmod{11}.
\end{align*}Finally, $11 \cdot 9 = 99$ and $108 - 99 = 9$, so $108 \equiv 9 \pmod{11}$. As a check, the powers computed above give $3^4 + 3^3 \equiv 4 + 5 = 9 \pmod{11}$, which agrees.

Therefore, the remainder is $\boxed{9}$.""",
    "test/number_theory/612.json": r"""Suppose Zach puts $n$ pencils into each bag and has $r$ pencils left over. Each bag must hold the greatest number of pencils possible, so there can be no way to put one more pencil into every bag while keeping the three bags equal.

Putting one more pencil into each of the three bags uses exactly $3$ pencils. If Zach has three or more pencils left over, that is, if $r \ge 3$, then he can take $3$ of the leftover pencils and add another pencil to each bag. Each bag would then hold $n + 1$ pencils, the three bags would still hold the same number, and he would have $r - 3$ pencils left over. That contradicts the requirement that each bag hold the greatest number possible. Therefore $r \le 2$.

Next we check that $2$ pencils left over can actually happen. Suppose Zach has $8$ pencils. He can put $2$ pencils into each bag, using $3 \cdot 2 = 6$ pencils, and have $8 - 6 = 2$ left over. He cannot put $3$ pencils into each bag, because that would need $3 \cdot 3 = 9$ pencils and he has only $8$. So with $8$ pencils, $2$ are left over.

In other words, the number of pencils left over is the remainder when the total number of pencils is divided by $3$, and a remainder upon division by $3$ can only be $0$, $1$, or $2$. The largest of these is $2$.

Therefore, Zach can have at most $\boxed{2}$ pencils left over.""",
}

SOLUTIONS_SHA256 = "f41c196505cc241a78f372f60123f99a0c6b56f92880ecb5d338bab8b987bfa8"
assert hashlib.sha256((json.dumps(SOLUTIONS, indent=1, ensure_ascii=False) + "\n").encode()).hexdigest() == SOLUTIONS_SHA256
print("worked solutions sha256:", SOLUTIONS_SHA256)

## Prompts

Both prompts end the same way. The few-shot prompt puts the five worked examples, levels 2, 4, 3, 5 and 1, in front. Every prompt's length in tokens is saved as `prompt_tokens.json`.

In [ ]:
FEW_SHOT_PREFIX = "".join(
    f"Problem:\n{r['problem']}\n\nSolution:\n{SOLUTIONS[r['unique_id']]}\n\n" for r in examples
)

def build_prompt(arm, problem):
    question = f"Problem:\n{problem}\n\nSolution:\n"
    return question if arm == "zero_shot" else FEW_SHOT_PREFIX + question

tok = AutoTokenizer.from_pretrained(MODELS[0])
print(f"few-shot prefix: {len(tok.encode(FEW_SHOT_PREFIX))} tokens before the test problem")

PROMPT_TOKENS = {arm: {r["unique_id"]: len(tok(build_prompt(arm, r["problem"]))["input_ids"]) for r in test_rows} for arm in ARMS}
with open(os.path.join(OUT_DIR, "prompt_tokens.json"), "w") as f:
    json.dump(PROMPT_TOKENS, f, indent=1)
print("median prompt tokens:", {arm: statistics.median(v.values()) for arm, v in PROMPT_TOKENS.items()})

In [ ]:
print(build_prompt("few_shot", test_rows[0]["problem"]))

## Grading

The first `\boxed{}` in a reply is its answer, compared with `math_verify`. Text after a new `Problem:` is ignored. A reply with no box is wrong.

In [ ]:
def extract_answer(text):
    for phrase in ["Difficulty Level:", "Problem:"]:
        if phrase in text:
            text = text.split(phrase)[0]
    if "\\boxed{" not in text:
        return ""
    start = text.find("\\boxed{") + len("\\boxed{")
    depth = 1
    for i in range(start, len(text)):
        if text[i] == "{":
            depth += 1
        elif text[i] == "}":
            depth -= 1
        if depth == 0:
            return text[start:i].strip()
    return ""

def grade(pred, gold):
    if pred == "":
        return False
    try:
        return bool(verify(parse(f"${gold}$"), parse(f"${pred}$")))
    except Exception:
        return pred.replace(" ", "").lower() == gold.replace(" ", "").lower()

assert grade(extract_answer(r"so $x=\boxed{\frac{1}{2}}$"), r"\frac12")
assert not grade(extract_answer(r"The answer is 12."), "12")
assert not grade(extract_answer(r"$\boxed{5}$ Problem: next $\boxed{6}$"), "6")
for r in examples:
    assert grade(extract_answer(SOLUTIONS[r["unique_id"]]), r["answer"]), r["unique_id"]

## Generation

Greedy decoding, stopping when the model starts a new `Problem:`. A reply's token count runs up to its first end or padding token. A reply that reaches the token limit without stopping is marked truncated.

In [ ]:
@torch.inference_mode()
def generate(model, tok, prompts):
    enc = tok(prompts, return_tensors="pt", padding=True).to(model.device)
    out = model.generate(
        **enc,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=False, temperature=None, top_p=None, top_k=None,
        repetition_penalty=1.0,
        pad_token_id=tok.pad_token_id,
        stop_strings=["Problem:"], tokenizer=tok,
    )
    end_ids = {tok.pad_token_id, tok.eos_token_id}
    replies = []
    for ids in out[:, enc["input_ids"].shape[1]:].tolist():
        n = next((k for k, t in enumerate(ids) if t in end_ids), len(ids))
        text = tok.decode(ids[:n], skip_special_tokens=True)
        replies.append({"reply": text, "tokens": n,
                        "truncated": n >= MAX_NEW_TOKENS and "Problem:" not in text})
    return replies

## Run

Replies already in `traces.jsonl` are skipped, so this cell can be run again after a disconnect.

In [ ]:
def load_traces():
    records = {}
    if os.path.exists(TRACES):
        with open(TRACES) as f:
            for line in f:
                try:
                    r = json.loads(line)
                except json.JSONDecodeError:
                    continue  # a line cut off by a disconnect
                records[(r["model"], r["arm"], r["unique_id"])] = r
    return records

if os.path.exists(TRACES) and os.path.getsize(TRACES):
    with open(TRACES, "rb") as f:
        f.seek(-1, os.SEEK_END)
        ends_clean = f.read(1) == b"\n"
    if not ends_clean:
        with open(TRACES, "a") as f:
            f.write("\n")  # end a line cut off by a disconnect, so new replies start on their own line

done = load_traces()
print(len(done), "replies already saved")

for model_id in MODELS:
    todo = {arm: [r for r in test_rows if (model_id, arm, r["unique_id"]) not in done] for arm in ARMS}
    if not any(todo.values()):
        print(model_id, "already complete")
        continue

    tok = AutoTokenizer.from_pretrained(model_id)
    tok.padding_side = "left"
    model = AutoModelForCausalLM.from_pretrained(model_id, dtype=torch.bfloat16, device_map="cuda")
    model.eval()

    for arm in ARMS:
        items, i, bs, start = todo[arm], 0, BATCH_SIZE, time.time()
        while i < len(items):
            batch = items[i:i + bs]
            try:
                replies = generate(model, tok, [build_prompt(arm, r["problem"]) for r in batch])
            except torch.cuda.OutOfMemoryError:
                torch.cuda.empty_cache()
                bs = max(1, bs // 2)
                print(f"  out of memory, batch size now {bs}")
                continue
            with open(TRACES, "a") as f:
                for r, g in zip(batch, replies):
                    extracted = extract_answer(g["reply"])
                    f.write(json.dumps({
                        "model": model_id, "arm": arm, "unique_id": r["unique_id"], "level": r["level"],
                        "gold": r["answer"], "reply": g["reply"], "extracted": extracted,
                        "correct": grade(extracted, r["answer"]), "tokens": g["tokens"],
                        "truncated": g["truncated"], "boxed": "\\boxed{" in g["reply"].split("Problem:")[0],
                        "wrote_code": "```python" in g["reply"], "batch_size": bs,
                    }, ensure_ascii=False) + "\n")
                f.flush()
                os.fsync(f.fileno())
            i += len(batch)
            print(f"  {model_id} {arm}: {i}/{len(items)}  {time.time() - start:.0f}s")

    del model
    gc.collect()
    torch.cuda.empty_cache()

## Results

Accuracy overall and by level for each model and prompt, with median prompt and reply length and counts of truncated, unboxed and code-writing replies. Saved to Drive as `summary.json`.

In [ ]:
records = load_traces()
test_ids = {r["unique_id"] for r in test_rows}
summary = []
for model_id in MODELS:
    for arm in ARMS:
        rs = [r for (m, a, u), r in records.items() if m == model_id and a == arm and u in test_ids]
        if not rs:
            continue
        by_level = {}
        for lvl in range(1, 6):
            lr = [r for r in rs if r["level"] == lvl]
            by_level[lvl] = {"n": len(lr), "correct": sum(r["correct"] for r in lr),
                             "median_tokens": statistics.median(r["tokens"] for r in lr) if lr else None}
        summary.append({
            "model": model_id, "arm": arm, "n": len(rs), "correct": sum(r["correct"] for r in rs),
            "accuracy": sum(r["correct"] for r in rs) / len(rs),
            "median_tokens": statistics.median(r["tokens"] for r in rs),
            "median_prompt_tokens": statistics.median(PROMPT_TOKENS[arm][r["unique_id"]] for r in rs),
            "truncated": sum(r["truncated"] for r in rs), "unboxed": sum(not r["boxed"] for r in rs),
            "wrote_code": sum(r["wrote_code"] for r in rs), "by_level": by_level,
            "solutions_sha256": SOLUTIONS_SHA256,
        })

with open(os.path.join(OUT_DIR, "summary.json"), "w") as f:
    json.dump(summary, f, indent=2)

print(f"{'model':20} {'arm':10} {'acc':>6} {'med tok':>8} {'trunc':>6} {'unboxed':>8} {'code':>5}   accuracy by level 1-5")
for s in summary:
    lv = "  ".join(f"{100 * v['correct'] / v['n']:5.1f}" if v["n"] else "    -" for v in s["by_level"].values())
    print(f"{s['model']:20} {s['arm']:10} {100 * s['accuracy']:5.1f}% {s['median_tokens']:8} {s['truncated']:6} {s['unboxed']:8} {s['wrote_code']:5}   {lv}")

## Paired comparison

On the same problems: how many each prompt solves that the other does not.

In [ ]:
for model_id in MODELS:
    pairs = [(records.get((model_id, "zero_shot", u)), records.get((model_id, "few_shot", u))) for u in sorted(test_ids)]
    pairs = [(z, f) for z, f in pairs if z and f]
    if not pairs:
        continue
    both = sum(z["correct"] and f["correct"] for z, f in pairs)
    zero_only = sum(z["correct"] and not f["correct"] for z, f in pairs)
    few_only = sum(f["correct"] and not z["correct"] for z, f in pairs)
    print(f"{model_id:20} n={len(pairs)}  both {both}  zero-shot only {zero_only}  few-shot only {few_only}")

## Charts

Overall accuracy for each model, then accuracy by level. Both are saved to Drive beside the traces.

In [ ]:
COLORS = {"zero_shot": "#4C72B0", "few_shot": "#DD8452"}
LABELS = {"zero_shot": "zero-shot", "few_shot": "few-shot CoT"}
acc = {(s["model"], s["arm"]): 100 * s["accuracy"] for s in summary}

fig, ax = plt.subplots(figsize=(8, 5))
for k, arm in enumerate(ARMS):
    xs = [i + (k - 0.5) * 0.38 for i in range(len(MODELS))]
    ys = [acc.get((m, arm), 0) for m in MODELS]
    bars = ax.bar(xs, ys, width=0.38, color=COLORS[arm], label=LABELS[arm])
    ax.bar_label(bars, fmt="%.1f%%", padding=3)
ax.set_xticks(range(len(MODELS)))
ax.set_xticklabels([m.split("/")[-1] for m in MODELS])
ax.set_ylabel("accuracy (%)")
ax.set_ylim(0, 100)
ax.set_title(f"Qwen2.5 base models on {len(test_rows)} MATH-500 problems")
ax.legend()
fig.tight_layout()
fig.savefig(os.path.join(OUT_DIR, "accuracy.png"), dpi=150)
plt.show()

fig, axes = plt.subplots(1, len(MODELS), figsize=(5 * len(MODELS), 4.5), sharey=True, squeeze=False)
for ax, model_id in zip(axes[0], MODELS):
    for k, arm in enumerate(ARMS):
        s = next((x for x in summary if x["model"] == model_id and x["arm"] == arm), None)
        if s is None:
            continue
        ys = [100 * v["correct"] / v["n"] if v["n"] else 0 for v in s["by_level"].values()]
        ax.bar([lvl + (k - 0.5) * 0.4 for lvl in range(1, 6)], ys, width=0.4, color=COLORS[arm], label=LABELS[arm])
    ax.set_title(model_id.split("/")[-1])
    ax.set_xlabel("MATH level")
    ax.set_xticks(range(1, 6))
    ax.legend(fontsize=9)
axes[0][0].set_ylabel("accuracy (%)")
axes[0][0].set_ylim(0, 100)
fig.suptitle("Accuracy by level")
fig.tight_layout()
fig.savefig(os.path.join(OUT_DIR, "accuracy_by_level.png"), dpi=150)
plt.show()

print("Saved to", OUT_DIR, ":", sorted(os.listdir(OUT_DIR)))